In [1]:
import numpy as np
import pandas as pd
import math

In [2]:
# import synthetic landscape

data_folder = ''

lscape_name_notxt = 'FERM_zone0'

img_folder = '' + lscape_name_notxt + '/'

synscape = pd.read_csv(data_folder + lscape_name_notxt + '.txt', sep=' ')
print(synscape.shape)
print(synscape.columns)

(211229, 33)
Index(['xpos', 'ypos', 'zpos', 'tens', 'fric', 'coh', 'C:Tau', 'ssr', 'ssi',
       'smax', 'smid', 'smin', 'sxx', 'sxy', 'sxz', 'syy', 'syz', 'szz', 'vsi',
       'vsr', 'xxr', 'yyr', 'zzr', 'xyr', 'yzr', 'xzr', 'xxi', 'yyi', 'zzi',
       'xyi', 'yzi', 'xzi', 'diff-str'],
      dtype='object')


In [3]:
# investigate the data structure

print("spatial ranges")
print(f"xpos: {synscape['xpos'].min():.2f} to {synscape['xpos'].max():.2f}")
print(f"ypos: {synscape['ypos'].min():.2f} to {synscape['ypos'].max():.2f}")
print(f"zpos: {synscape['zpos'].min():.2f} to {synscape['zpos'].max():.2f}")

print(f"\nunique x: {synscape['xpos'].nunique()}")
print(f"unique y: {synscape['ypos'].nunique()}")
print(f"unique z: {synscape['zpos'].nunique()}")

print(f"\ntotal points: {len(synscape)}")
print(f"unique z * unique y * unique x would be: "
      f"{synscape['xpos'].nunique() * synscape['ypos'].nunique() * synscape['zpos'].nunique()}")

spatial ranges
xpos: 24.93 to 4981.94
ypos: 24.38 to 4923.50
zpos: -3748.31 to 963.20

unique x: 38159
unique y: 18504
unique z: 152577

total points: 211229
unique z * unique y * unique x would be: 107733724988472


In [4]:
# Claude: not a regular 3D grid
# scattered in 3D space rather than lattice
# consistent with particle-based / finite element simulation

In [5]:
# further analysis
# how many points share the same x,y location (i.e. vertical columns)?
xy_groups = synscape.groupby(['xpos', 'ypos']).size()
print("points per x,y location:")
print(xy_groups.describe())
print(f"x,y locations with exactly 1 point: {(xy_groups == 1).sum()}")
print(f"x,y locations with > 1 point: {(xy_groups > 1).sum()}")

# what does the z distribution look like?
print(f"\nzpos percentiles:")
for p in [1, 5, 25, 50, 75, 95, 99]:
    print(f"  {p}th: {synscape['zpos'].quantile(p/100):.2f}")

# are there natural depth layers?
print(f"\nzpos value counts (top 20 most common):")
print(synscape['zpos'].round(0).value_counts().head(20))

points per x,y location:
count    200605.000000
mean          1.052960
std           0.319653
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          13.000000
dtype: float64
x,y locations with exactly 1 point: 192913
x,y locations with > 1 point: 7692

zpos percentiles:
  1th: -3727.57
  5th: -3240.68
  25th: -1520.59
  50th: -469.01
  75th: -80.46
  95th: 300.86
  99th: 477.82

zpos value counts (top 20 most common):
zpos
-2.0       175
-13.0      173
-3.0       172
-7.0       172
-8.0       171
-6.0       171
-16.0      171
-1.0       170
-51.0      169
-21.0      169
-11.0      168
-4.0       168
-3744.0    168
-52.0      168
-23.0      167
-41.0      167
-32.0      167
-49.0      167
-10.0      167
-20.0      167
Name: count, dtype: int64


In [6]:
# possibility: non-unique x-y points "overlap"
# due to particle-based flow

In [7]:
# investigate x-y spatial coverage
# bin into a coarse grid to see spatial coverage
bin_size = 60  # 100 unit bins

synscape['xbin'] = (synscape['xpos'] // bin_size).astype(int)
synscape['ybin'] = (synscape['ypos'] // bin_size).astype(int)

xy_bin_counts = synscape.groupby(['xbin', 'ybin']).size()
print(f"bin size: {bin_size}")
print(f"\ncoarse grid bins occupied: {len(xy_bin_counts)}")
print(f"points per coarse bin:")
print(xy_bin_counts.describe())

# now look at z distribution within near-surface points only
min_zpos = -1
near_surface = synscape[synscape['zpos'] > min_zpos]
print(f"\nmin_zpos: {min_zpos}")
print(f"\nnear-surface points (zpos > {min_zpos}): {len(near_surface)}")
print(f"unique x,y in near-surface: {near_surface.groupby(['xpos','ypos']).ngroups}")
print(f"\nnear-surface zpos distribution:")
print(near_surface['zpos'].describe())

bin size: 60

coarse grid bins occupied: 6838
points per coarse bin:
count    6838.000000
mean       30.890465
std        15.319822
min        14.000000
25%        21.000000
50%        25.000000
75%        40.000000
max       108.000000
dtype: float64

min_zpos: -1

near-surface points (zpos > -1): 39943
unique x,y in near-surface: 36066

near-surface zpos distribution:
count    39943.000000
mean       208.389874
std        148.910929
min         -0.989259
25%         84.421000
50%        184.407000
75%        308.815000
max        963.196000
Name: zpos, dtype: float64


In [8]:
# for each coarse bin, look at the z range of points
z_range_per_bin = synscape.groupby(['xbin', 'ybin'])['zpos'].agg(['min', 'max', 'count'])
z_range_per_bin['z_range'] = z_range_per_bin['max'] - z_range_per_bin['min']

print("z range per coarse bin:")
print(z_range_per_bin['z_range'].describe())
print(f"\nbins with z_range > 100: {(z_range_per_bin['z_range'] > 100).sum()}")
print(f"bins with z_range > 500: {(z_range_per_bin['z_range'] > 500).sum()}")
print(f"bins with z_range > 1000: {(z_range_per_bin['z_range'] > 1000).sum()}")

# what does the material property look like at different depth bands?
depth_bands = [(-1, 100), (100, 500), (500, 1000), (1000, 2000), (2000, 5000)]
print("\nMoran's I by depth band -- cohesion:")
for z_lo, z_hi in depth_bands:
    # depth below surface -- use negative zpos as proxy for depth
    band = synscape[(synscape['zpos'] >= -z_hi) & (synscape['zpos'] < -z_lo)]
    if len(band) > 100:
        print(f"  zpos [{-z_hi:.0f} to {-z_lo:.0f}]: {len(band)} points, "
              f"coh mean={band['coh'].mean():.2f}, std={band['coh'].std():.2f}")

z range per coarse bin:
count    6838.000000
mean     3872.400183
std       227.143374
min      3214.957000
25%      3696.279600
50%      3837.493200
75%      4063.382500
max      4711.376000
Name: z_range, dtype: float64

bins with z_range > 100: 6838
bins with z_range > 500: 6838
bins with z_range > 1000: 6838

Moran's I by depth band -- cohesion:
  zpos [-100 to 1]: 16196 points, coh mean=878729.49, std=565864.86
  zpos [-500 to -100]: 52853 points, coh mean=963829.28, std=533622.61
  zpos [-1000 to -500]: 31559 points, coh mean=966858.48, std=530501.79
  zpos [-2000 to -1000]: 32005 points, coh mean=981038.85, std=526321.57
  zpos [-5000 to -2000]: 38968 points, coh mean=989978.11, std=524170.53
